# External Model Baselines — Demo with Dummy Input

This notebook walks through the three external feature extractors step by step using synthetic multiplexed cell patches — no real data or downloaded weights needed.

**Models covered:**
1. **DINOv2** — Meta self-supervised ViT, channel-agnostic adaptation (auto-downloads ~330 MB)
2. **OpenPhenom** — Recursion channel-agnostic MAE, natively multi-channel (auto-downloads ~100 MB)
3. **UNI** — Mahmood Lab pathology ViT, same adaptation as DINOv2 (requires gated HuggingFace access)

**What we test:** input/output shapes, preprocessing, the channel-agnostic patch embed, and feature statistics — all with random `[B, C, H, W]` tensors that mimic real cell patches.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))   # models_external.py lives here

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {DEVICE}")

## 1 — Dummy multiplexed cell patches

We simulate a batch of `B=4` cells from a 18-marker panel (KRONOS18 size), each patch `32×32` pixels — exactly what the MCA datasets produce.

In [ ]:
torch.manual_seed(0)

B          = 4    # batch size
C_kronos   = 18   # KRONOS18 markers
C_full     = 41   # full cHL panel
C_mibi     = 37   # MIBI-TNBC panel
H, W       = 32, 32  # patch size

# Patches in [0, 1] float32 — what MCIDatasetH5 returns after normalisation
patches_18 = torch.rand(B, C_kronos, H, W)
patches_41 = torch.rand(B, C_full,   H, W)
patches_37 = torch.rand(B, C_mibi,   H, W)

print(f"18-marker batch : {patches_18.shape}  dtype={patches_18.dtype}")
print(f"41-marker batch : {patches_41.shape}  dtype={patches_41.dtype}")
print(f"37-marker batch : {patches_37.shape}  dtype={patches_37.dtype}")

# Visualise one cell: each row = one marker channel
fig, axes = plt.subplots(3, 6, figsize=(12, 6))
fig.suptitle("Dummy 18-marker cell patch (batch[0], first 18 channels)", fontsize=12)
for i, ax in enumerate(axes.flat):
    ax.imshow(patches_18[0, i].numpy(), cmap='inferno', vmin=0, vmax=1)
    ax.set_title(f"Ch {i}", fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 2 — Shared preprocessing utilities

Both DINOv2 and UNI need two things before the ViT sees the data:
- **Resize** from 32×32 → 224×224 (bilinear)
- **Per-channel instance normalisation** — zero-mean, unit-std per marker per cell (removes absolute intensity variation across patients)

In [ ]:
from models_external import _per_channel_instance_norm

# ── Resize ────────────────────────────────────────────────────────────────────
x_resized = F.interpolate(patches_18, size=(224, 224), mode='bilinear', align_corners=False)
print(f"After resize  : {x_resized.shape}")   # [4, 18, 224, 224]

# ── Per-channel instance norm ────────────────────────────────────────────────
x_normed = _per_channel_instance_norm(x_resized)
print(f"After norm    : {x_normed.shape}")

# Check: each channel of each cell should now be ~zero-mean, ~unit-std
cell0 = x_normed[0]   # [18, 224, 224]
means = cell0.reshape(C_kronos, -1).mean(dim=1)
stds  = cell0.reshape(C_kronos, -1).std(dim=1)
print(f"\nCell 0 — per-channel mean : min={means.min():.4f}  max={means.max():.4f}  (should be ~0)")
print(f"Cell 0 — per-channel std  : min={stds.min():.4f}   max={stds.max():.4f}   (should be ~1)")

# Compare raw vs normalised for one marker
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for i, (ax, img, title) in enumerate(zip(axes,
    [patches_18[0, 0], patches_18[1, 0], x_normed[0, 0], x_normed[1, 0]],
    ['Raw cell 0, ch0', 'Raw cell 1, ch0', 'Normed cell 0, ch0', 'Normed cell 1, ch0'])):
    im = ax.imshow(img.detach().numpy(), cmap='RdBu_r')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.suptitle("Raw vs per-channel instance-normalised patches", fontsize=11)
plt.tight_layout()
plt.show()

## 3 — Channel-agnostic patch embedding

The key adaptation for DINOv2 and UNI: replace the 3-channel patch projection with a single-channel one, then apply it independently to each marker.

**Standard ViT patch embed** (3-channel RGB):
```
Conv2d(3, D, patch_size, stride=patch_size)  →  [B, N, D]
```

**Channel-agnostic** (C markers):
```
Conv2d(1, D, patch_size, stride=patch_size) applied to each marker  →  C × [B, N, D]
concat  →  [B, C*N, D]
```

The new Conv2d weights are initialised as the mean of the 3 pretrained RGB filters — no random init, structure is preserved.

In [ ]:
from models_external import _adapt_patch_embed_to_1channel

# Simulate the pretrained 3-channel patch projection from DINOv2-ViT-B/14
# (D=768, patch_size=14)
D, patch_size = 768, 14
original_proj = nn.Conv2d(3, D, patch_size, stride=patch_size)  # pretrained (random here)
print(f"Original proj weight : {original_proj.weight.shape}")   # [768, 3, 14, 14]

# Adapt to 1-channel
adapted_proj = _adapt_patch_embed_to_1channel(original_proj)
print(f"Adapted  proj weight : {adapted_proj.weight.shape}")    # [768, 1, 14, 14]

# Verify: adapted weight == mean over RGB dim of original
expected = original_proj.weight.mean(dim=1, keepdim=True)
diff = (adapted_proj.weight - expected).abs().max().item()
print(f"Max weight diff (should be 0): {diff:.6f}")

# ── Show how tokens are produced per channel ─────────────────────────────────
x_small = x_normed[:, :, :28, :28]   # use 28×28 for quick demo → 2×2 = 4 patches/channel

all_tokens = []
for c in range(C_kronos):
    tok = adapted_proj(x_small[:, c:c+1])        # [B, D, h_p, w_p]
    tok = tok.flatten(2).transpose(1, 2)          # [B, N, D]
    all_tokens.append(tok)

tokens = torch.cat(all_tokens, dim=1)             # [B, C*N, D]
print(f"\nPatch tokens per channel : {all_tokens[0].shape}  (B, N, D)")
print(f"All channels concatenated : {tokens.shape}  (B, C*N, D)")
print(f"  N = {all_tokens[0].shape[1]} spatial patches per marker")
print(f"  C*N = {C_kronos} × {all_tokens[0].shape[1]} = {tokens.shape[1]} total tokens")

## 4 — DINOv2 with a mock ViT (no download needed)

Before loading real pretrained weights, let's verify the full channel-agnostic forward pass with a tiny randomly-initialised ViT that has the same structure as DINOv2-ViT-B/14 but tiny dimensions — runs instantly on CPU.

In [ ]:
from models_external import _channel_agnostic_vit_forward

# ── Build a tiny mock ViT that mirrors DINOv2's attribute structure ──────────
D_mock     = 64     # embed dim (real DINOv2-B = 768)
patch_size = 14
img_size   = 224
N_patches  = (img_size // patch_size) ** 2   # 16*16 = 256

class MockDINO(nn.Module):
    """Minimal DINOv2-shaped ViT for testing channel-agnostic forward."""
    def __init__(self):
        super().__init__()

        class PatchEmbed(nn.Module):
            def __init__(self):
                super().__init__()
                self.proj = nn.Conv2d(1, D_mock, patch_size, stride=patch_size)

        self.patch_embed     = PatchEmbed()
        self.cls_token       = nn.Parameter(torch.zeros(1, 1, D_mock))
        # pos_embed: CLS token + N spatial patches
        self.pos_embed       = nn.Parameter(torch.randn(1, 1 + N_patches, D_mock) * 0.02)
        self.register_tokens = None           # no register tokens in this mock
        self.blocks          = nn.Sequential(
            nn.TransformerEncoderLayer(d_model=D_mock, nhead=4, dim_feedforward=D_mock*4,
                                       batch_first=True, norm_first=True)
        )
        self.norm = nn.LayerNorm(D_mock)

mock_vit = MockDINO()
total_params = sum(p.numel() for p in mock_vit.parameters())
print(f"Mock ViT parameters : {total_params:,}  (real DINOv2-B has ~86M)")
print(f"N spatial patches   : {N_patches} per marker channel")

# ── Run channel-agnostic forward ─────────────────────────────────────────────
x_in = patches_18                             # [4, 18, 32, 32]
x_resized_224 = F.interpolate(x_in, size=(224, 224), mode='bilinear', align_corners=False)
x_normed_224  = _per_channel_instance_norm(x_resized_224)

with torch.no_grad():
    cls_feat = _channel_agnostic_vit_forward(mock_vit, x_normed_224, model_type='dinov2')

print(f"\nInput  : {x_in.shape}  →  resized {x_resized_224.shape}")
print(f"Output (CLS token) : {cls_feat.shape}")   # [B, D_mock]
print(f"\nFeature stats (random weights, not meaningful):")
print(f"  mean={cls_feat.mean().item():.4f}  std={cls_feat.std().item():.4f}")

## 5 — DINOv2 with real pretrained weights

This cell downloads DINOv2-ViT-B/14 (~330 MB, only once) and runs the full channel-agnostic forward pass on all three dummy panels.

In [ ]:
from models_external import DINOv2Backbone

# Downloads weights on first run (~330 MB), cached in torch hub after that
dinov2 = DINOv2Backbone(variant='dinov2_vitb14', img_size=224, freeze=True)
dinov2 = dinov2.to(DEVICE)
dinov2.eval()

n_params = sum(p.numel() for p in dinov2.parameters())
print(f"DINOv2-ViT-B/14 parameters : {n_params/1e6:.1f} M")
print(f"Output embedding dim       : {dinov2.out_channels}")

print("\nRunning channel-agnostic forward pass...")
results = {}
with torch.no_grad():
    for name, patches in [('18-marker (KRONOS18)', patches_18),
                           ('41-marker (cHL full)', patches_41),
                           ('37-marker (MIBI-TNBC)', patches_37)]:
        feat = dinov2(patches.to(DEVICE))[0]           # [B, D, 1, 1]
        feat = feat.squeeze(-1).squeeze(-1)             # [B, D]
        results[name] = feat.cpu()
        print(f"  {name:<26}  input={tuple(patches.shape)}  →  feat={tuple(feat.shape)}")

In [ ]:
# Visualise the DINOv2 embeddings: pairwise cosine similarity between the 4 dummy cells
from sklearn.metrics.pairwise import cosine_similarity

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle("DINOv2-ViT-B/14 — pairwise cosine similarity (4 dummy cells)", fontsize=12)

for ax, (name, feat) in zip(axes, results.items()):
    feat_np = feat.numpy()
    sim = cosine_similarity(feat_np)
    im = ax.imshow(sim, vmin=-1, vmax=1, cmap='RdBu_r')
    ax.set_title(name, fontsize=9)
    ax.set_xticks(range(B)); ax.set_yticks(range(B))
    ax.set_xticklabels([f'cell {i}' for i in range(B)], fontsize=8)
    ax.set_yticklabels([f'cell {i}' for i in range(B)], fontsize=8)
    for i in range(B):
        for j in range(B):
            ax.text(j, i, f'{sim[i,j]:.2f}', ha='center', va='center', fontsize=9)
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.show()

print("\nNote: all similarities are close to 1 because the input patches are random uniform noise —")
print("the model can't distinguish them. Real cell patches will show much more variation.")

## 6 — OpenPhenom (Recursion)

OpenPhenom needs **no weight adaptation** — it was trained natively with a channel-agnostic `Conv2d(1→384)` on 6-channel Cell Painting microscopy data, and supports up to 11 channels at inference. We just resize to 256×256 and call `predict()`.

Weights download automatically from `recursionpharma/OpenPhenom` on HuggingFace (~100 MB).

In [ ]:
from models_external import OpenPhenomBackbone

# Downloads weights from recursionpharma/OpenPhenom on first run (~100 MB)
phenom = OpenPhenomBackbone(img_size=256, freeze=True).to(DEVICE)
phenom.eval()

n_params = sum(p.numel() for p in phenom.parameters())
print(f"OpenPhenom parameters : {n_params/1e6:.1f} M")
print(f"Output embedding dim  : {phenom.out_channels}")

# Inspect the channel-agnostic patch embed — Conv2d(1→384)
enc = phenom.model.encoder.vit_backbone
print(f"\nPatch embed proj : {enc.patch_embed.proj}")   # Conv2d(1, 384, ...)
print(f"Pos embed shape  : {enc.pos_embed.shape}")      # [1, 1 + max_chans*N, 384]

In [ ]:
print("Running OpenPhenom on all three dummy panels...")
phenom_results = {}

with torch.no_grad():
    for name, patches in [('18-marker (KRONOS18)', patches_18),
                           ('41-marker (cHL full)', patches_41),
                           ('37-marker (MIBI-TNBC)', patches_37)]:
        feat = phenom(patches.to(DEVICE))[0]        # [B, 384, 1, 1]
        feat = feat.squeeze(-1).squeeze(-1)          # [B, 384]
        phenom_results[name] = feat.cpu()
        print(f"  {name:<26}  input={tuple(patches.shape)}  →  feat={tuple(feat.shape)}")

# OpenPhenom also supports per-channel embeddings (384 per marker)
print("\nPer-channel embeddings mode (return_channelwise_embeddings=True):")
phenom.model.return_channelwise_embeddings = True
with torch.no_grad():
    x_u8 = (patches_18 * 255).clamp(0, 255).to(torch.uint8).to(DEVICE)
    x_u8 = F.interpolate(x_u8.float(), size=(256, 256), mode='bilinear',
                          align_corners=False).to(torch.uint8)
    feat_per_ch = phenom.model.predict(x_u8)   # [B, 384*C]
    print(f"  input={tuple(patches_18.shape)}  →  per-channel feat={tuple(feat_per_ch.shape)}")
    print(f"  = B × (384 × {C_kronos}) = {B} × {384 * C_kronos}")
phenom.model.return_channelwise_embeddings = False   # reset

## 7 — UNI (Mahmood Lab)

UNI is a ViT-L/16 trained on 200 million H&E and IHC pathology images. It uses the **same channel-agnostic adaptation** as DINOv2: the pretrained 3-channel patch projection is replaced by a 1-channel version initialised from the RGB weight average, and all transformer weights remain frozen.

**Key difference from DINOv2:** UNI weights are **gated** on HuggingFace — you must request access at [MahmoodLab/UNI](https://huggingface.co/MahmoodLab/UNI) before downloading. The cell below walks through the architecture without loading real weights (to keep the notebook runnable without a HF token). At the end we show how to load real weights when available.

UNI architecture (ViT-L/16):
- Patch size: 16×16
- Embed dim D: 1024
- Depth: 24 transformer blocks
- Heads: 16
- ~307M parameters

In [ ]:
import timm

# Build UNI with random weights (same architecture, no checkpoint needed)
uni_kwargs = dict(
    model_name='vit_large_patch16_224',
    img_size=224,
    patch_size=16,
    init_values=1e-5,
    num_classes=0,
    dynamic_img_size=True,
)
uni_mock = timm.create_model(**uni_kwargs)
uni_mock.eval()

n_params_uni = sum(p.numel() for p in uni_mock.parameters())
print(f"UNI (ViT-L/16) parameters : {n_params_uni/1e6:.1f} M")
print(f"Original patch embed proj  : {uni_mock.patch_embed.proj}")
# Conv2d(3, 1024, kernel_size=(16, 16), stride=(16, 16))

from models_external import _adapt_patch_embed_to_1channel, _channel_agnostic_vit_forward

# Adapt: Conv2d(3→1024) → Conv2d(1→1024)
uni_mock.patch_embed.proj = _adapt_patch_embed_to_1channel(uni_mock.patch_embed.proj)
print(f"Adapted patch embed proj   : {uni_mock.patch_embed.proj}")

# Freeze
for p in uni_mock.parameters():
    p.requires_grad_(False)

# Run channel-agnostic forward with random-weight UNI
print("\nRunning UNI (random weights) channel-agnostic forward...")
with torch.no_grad():
    for name, patches in [('18-marker (KRONOS18)', patches_18),
                           ('41-marker (cHL full)', patches_41),
                           ('37-marker (MIBI-TNBC)', patches_37)]:
        B_cur = patches.shape[0]
        x = F.interpolate(patches, size=(224, 224), mode='bilinear', align_corners=False)
        x = _per_channel_instance_norm(x)
        cls = _channel_agnostic_vit_forward(uni_mock, x, model_type='timm')  # [B, 1024]
        feat_out = cls.view(B_cur, 1024, 1, 1)
        print(f"  {name:<26}  input={tuple(patches.shape)}  →  feat={tuple(feat_out.shape)}")

In [ ]:
# ── Loading real UNI weights (when you have HF access) ───────────────────────
# from models_external import UNIBackbone
#
# ckpt_path = '/path/to/MahmoodLab/UNI/pytorch_model.bin'  # after hf_hub_download
# uni = UNIBackbone(ckpt_path=ckpt_path, img_size=224, freeze=True).to(DEVICE)
# uni.eval()
#
# with torch.no_grad():
#     feat = uni(patches_18.to(DEVICE))[0]   # [B, 1024, 1, 1]
#
# Download instructions:
#   1. Request access at https://huggingface.co/MahmoodLab/UNI
#   2. pip install huggingface_hub
#   3. huggingface-cli login
#   4. python -c "from huggingface_hub import hf_hub_download; \
#        hf_hub_download('MahmoodLab/UNI', 'pytorch_model.bin', local_dir='./uni_weights')"

print("UNI real-weight loading: see comments above.")
print("Architecture is identical to the mock above — only the weights differ.")

## 8 — Summary comparison

| Model | Architecture | Training data | Params | Output dim | Patch size | Input size | Access |
|---|---|---|---|---|---|---|---|
| **DINOv2-ViT-B/14** | ViT-B | Natural images (LVD-142M) | ~86M | 768 | 14×14 | 224×224 | Free |
| **DINOv2-ViT-L/14** | ViT-L | Natural images (LVD-142M) | ~307M | 1024 | 14×14 | 224×224 | Free |
| **OpenPhenom** | ViT-S (MAE) | Cell Painting (6ch microscopy) | ~21M | 384 | 16×16 | 256×256 | Free |
| **UNI** | ViT-L | Pathology (200M H&E/IHC) | ~307M | 1024 | 16×16 | 224×224 | Gated HF |

**Channel adaptation strategy (DINOv2 and UNI):**
- Replace `Conv2d(3, D, p, p)` → `Conv2d(1, D, p, p)` initialised from `weight.mean(dim=1)`
- Apply 1-channel proj independently to each marker → C×N spatial tokens
- Tile positional embeddings C times (same spatial grid, different content)
- All transformer weights remain frozen

**OpenPhenom is different:** it was trained natively multi-channel, so no adaptation is needed — `predict()` accepts any number of channels out of the box.

**Preprocessing:**
- DINOv2 / UNI: resize → per-channel instance norm (zero-mean, unit-std per marker)
- OpenPhenom: resize to 256×256, convert to uint8 [0,255] (model self-normalises internally)

In [ ]:
# Summary: confirmed output shapes for all three models on all three panels
print("Model output shapes confirmed:")
print("-" * 65)
print(f"{'Model':<20}  {'Panel':<26}  Output shape")
print("-" * 65)

# DINOv2 results were stored in `results` dict above
for name, feat in results.items():
    print(f"{'DINOv2-ViT-B/14':<20}  {name:<26}  {tuple(feat.shape)}")

# OpenPhenom results were stored in `phenom_results`
for name, feat in phenom_results.items():
    print(f"{'OpenPhenom':<20}  {name:<26}  {tuple(feat.shape)}")

# UNI: same output shape as DINOv2-L (D=1024), shown from mock run
print(f"{'UNI (ViT-L/16)':<20}  {'18-marker (KRONOS18)':<26}  (4, 1024, 1, 1)  [random weights]")
print(f"{'UNI (ViT-L/16)':<20}  {'41-marker (cHL full)':<26}  (4, 1024, 1, 1)  [random weights]")
print(f"{'UNI (ViT-L/16)':<20}  {'37-marker (MIBI-TNBC)':<26}  (4, 1024, 1, 1)  [random weights]")

print("\nAll models expose the same MCA backbone interface:")
print("  Input:  [B, C, H, W]  float32  (any C)")
print("  Output: ([B, D, 1, 1],)  tuple")
print("\nRun tools/extract_external_features.py on a real .h5 file for actual baselines.")